In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix
)

# carga de datos
def cargar_datos_multiples_paises(rutas_archivos, nombres_paises):
    dataframes = []

    for ruta, pais in zip(rutas_archivos, nombres_paises):
        try:
            df = pd.read_csv(ruta, encoding='utf-8')
        except:
            df = pd.read_csv(ruta, encoding='latin-1')

        df['country'] = pais
        dataframes.append(df)

    df_completo = pd.concat(dataframes, ignore_index=True)
    return df_completo

rutas = [
    'USvideos.csv',
    'CAvideos.csv',
    'GBvideos.csv',
    'DEvideos.csv',
    'FRvideos.csv',
    'INvideos.csv',
    'JPvideos.csv',
    'KRvideos.csv',
    'MXvideos.csv',
    'RUvideos.csv'
]

paises = ['US', 'CA', 'GB', 'DE', 'FR', 'IN', 'JP', 'KR', 'MX', 'RU']

#archivo de US
df = pd.read_csv('USvideos.csv', encoding='latin-1')
df['country'] = 'US'

print(f"Dimensiones: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")

Dimensiones: (40949, 17)
Columnas: ['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description', 'country']


In [9]:
# procesamiento
def preprocesar_datos(df):
    df_clean = df.copy()

    #imputar valores nulos en 'description'
    if 'description' in df_clean.columns:
        df_clean['description'] = df_clean['description'].fillna('')

    #convertir fechas
    df_clean['trending_date'] = pd.to_datetime(
        df_clean['trending_date'],
        format='%y.%d.%m'
    )

    df_clean['publish_time'] = pd.to_datetime(df_clean['publish_time'])
    if df_clean['publish_time'].dt.tz is not None:
        df_clean['publish_time'] = df_clean['publish_time'].dt.tz_localize(None)

    #crear variables temporales
    df_clean['publish_date'] = df_clean['publish_time'].dt.date
    df_clean['publish_hour'] = df_clean['publish_time'].dt.hour
    df_clean['publish_dayofweek'] = df_clean['publish_time'].dt.dayofweek
    df_clean['publish_month'] = df_clean['publish_time'].dt.month
    df_clean['publish_year'] = df_clean['publish_time'].dt.year

    #calcular días hasta tendencia
    df_clean['days_to_trending'] = (
        df_clean['trending_date'] - df_clean['publish_time'].dt.normalize()
    ).dt.days

    #filtrar valores negativos o extremos
    df_clean = df_clean[df_clean['days_to_trending'] >= 0]
    df_clean = df_clean[df_clean['days_to_trending'] <= 365]  # Máximo 1 año

    #crear variable objetivo para clasificación
    df_clean['viral'] = (df_clean['views'] >= 1_000_000).astype(int)

    #métricas de engagement
    df_clean['engagement_rate'] = (
        (df_clean['likes'] + df_clean['dislikes'] + df_clean['comment_count'])
        / df_clean['views'].replace(0, 1)
    )

    df_clean['like_ratio'] = (
        df_clean['likes'] / (df_clean['likes'] + df_clean['dislikes']).replace(0, 1)
    )

    #transformación logarítmica para normalizar distribuciones
    for col in ['views', 'likes', 'dislikes', 'comment_count']:
        df_clean[f'log_{col}'] = np.log1p(df_clean[col])

    return df_clean

df = preprocesar_datos(df)
print(f"\nDimensiones después del preprocesamiento: {df.shape}")
print(f"\nEstadísticas de days_to_trending:")
print(df['days_to_trending'].describe())


Dimensiones después del preprocesamiento: (40712, 30)

Estadísticas de days_to_trending:
count    40712.000000
mean         6.875098
std          9.465539
min          0.000000
25%          3.000000
50%          5.000000
75%          8.000000
max        338.000000
Name: days_to_trending, dtype: float64


In [ ]:
# cargar categorias
import json

def cargar_categorias(ruta_json):
    with open(ruta_json, 'r') as f:
        data = json.load(f)

    categorias = {}
    for item in data['items']:
        categorias[int(item['id'])] = item['snippet']['title']

    return categorias
try:
    categorias = cargar_categorias('US_category_id.json')
    df['category_name'] = df['category_id'].map(categorias)
except:
    print("Archivo de categorías no encontrado. Usando category_id directamente.")
    df['category_name'] = df['category_id'].astype(str)

In [ ]:
# analisis exploratorio
def analisis_exploratorio(df):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    #distribución de vistas
    ax1 = axes[0, 0]
    ax1.hist(df['log_views'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.set_xlabel('Log(Vistas)')
    ax1.set_ylabel('Frecuencia')
    ax1.set_title('Distribución de Vistas (Escala Logarítmica)')

    #distribución de likes
    ax2 = axes[0, 1]
    ax2.hist(df['log_likes'], bins=50, edgecolor='black', alpha=0.7, color='green')
    ax2.set_xlabel('Log(Likes)')
    ax2.set_ylabel('Frecuencia')
    ax2.set_title('Distribución de Likes (Escala Logarítmica)')

    #distribución de comentarios
    ax3 = axes[1, 0]
    ax3.hist(df['log_comment_count'], bins=50, edgecolor='black', alpha=0.7, color='orange')
    ax3.set_xlabel('Log(Comentarios)')
    ax3.set_ylabel('Frecuencia')
    ax3.set_title('Distribución de Comentarios (Escala Logarítmica)')

    #días hasta tendencia
    ax4 = axes[1, 1]
    ax4.hist(df['days_to_trending'], bins=50, edgecolor='black', alpha=0.7, color='purple')
    ax4.set_xlabel('Días hasta Tendencia')
    ax4.set_ylabel('Frecuencia')
    ax4.set_title('Distribución de Días hasta Tendencia')

    plt.tight_layout()
    plt.savefig('distribucion_popularidad.png', dpi=150)
    plt.show()

    return fig

fig_eda = analisis_exploratorio(df)

In [ ]:
# analisis de patrones temporales
def analisis_patrones_temporales(df):

    #agregar día de la semana de tendencia
    df['trending_dayofweek'] = df['trending_date'].dt.dayofweek

    #calcular indice estacional por día
    videos_por_dia = df.groupby('trending_dayofweek').size()
    promedio_general = videos_por_dia.mean()
    indice_estacional = videos_por_dia / promedio_general

    dias_semana = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(dias_semana, indice_estacional.values, color='coral', edgecolor='black')
    ax.axhline(y=1, color='red', linestyle='--', label='Promedio (1.0)')
    ax.set_xlabel('Día de la semana')
    ax.set_ylabel('Índice estacional')
    ax.set_title('Índices estacionales de actividad en tendencias por día de la semana')
    ax.legend()

    #añadir valores sobre las barras
    for bar, val in zip(bars, indice_estacional.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig('patrones_estacionales.png', dpi=150)
    plt.show()

    return indice_estacional

indice_estacional = analisis_patrones_temporales(df)

In [ ]:
# features para los modelos
def preparar_features(df):

    #features relevantes
    features = [
        'category_id',
        'publish_hour',
        'publish_dayofweek',
        'publish_month',
        'log_views',
        'log_likes',
        'log_dislikes',
        'log_comment_count',
        'engagement_rate',
        'like_ratio',
        'comments_disabled',
        'ratings_disabled'
    ]

    #convertir booleanos a enteros
    df['comments_disabled'] = df['comments_disabled'].astype(int)
    df['ratings_disabled'] = df['ratings_disabled'].astype(int)

    #filtrar filas con valores válidos
    df_model = df.dropna(subset=features + ['viral', 'days_to_trending'])

    X = df_model[features]
    y_clasificacion = df_model['viral']
    y_regresion = df_model['days_to_trending']

    return X, y_clasificacion, y_regresion, df_model

X, y_clf, y_reg, df_model = preparar_features(df)
print(f"\nDimensiones de features: {X.shape}")
print(f"Distribución de clases (viral): {y_clf.value_counts().to_dict()}")

In [ ]:
# diviision temporal de los datos
def division_temporal(df_model, X, y_clf, y_reg):
    #ordenar por fecha de tendencia
    df_sorted = df_model.sort_values('trending_date').reset_index(drop=True)

    n = len(df_sorted)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    #indices para cada conjunto
    train_idx = df_sorted.index[:train_end]
    val_idx = df_sorted.index[train_end:val_end]
    test_idx = df_sorted.index[val_end:]

    #reindexar X, y_clf, y_reg con los mismos índices
    X_reindexed = X.reindex(df_sorted.index)
    y_clf_reindexed = y_clf.reindex(df_sorted.index)
    y_reg_reindexed = y_reg.reindex(df_sorted.index)

    X_train = X_reindexed.loc[train_idx]
    X_val = X_reindexed.loc[val_idx]
    X_test = X_reindexed.loc[test_idx]

    y_clf_train = y_clf_reindexed.loc[train_idx]
    y_clf_val = y_clf_reindexed.loc[val_idx]
    y_clf_test = y_clf_reindexed.loc[test_idx]

    y_reg_train = y_reg_reindexed.loc[train_idx]
    y_reg_val = y_reg_reindexed.loc[val_idx]
    y_reg_test = y_reg_reindexed.loc[test_idx]

    print(f"Entrenamiento: {len(X_train)} registros")
    print(f"Validación: {len(X_val)} registros")
    print(f"Prueba: {len(X_test)} registros")

    return (X_train, X_val, X_test,
            y_clf_train, y_clf_val, y_clf_test,
            y_reg_train, y_reg_val, y_reg_test)

(X_train, X_val, X_test,
 y_clf_train, y_clf_val, y_clf_test,
 y_reg_train, y_reg_val, y_reg_test) = division_temporal(df_model, X, y_clf, y_reg)

#escalar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# modelos de clasificación
def entrenar_modelos_clasificacion(X_train, X_test, y_train, y_test):
    modelos = {
        'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
    }

    resultados = []

    for nombre, modelo in modelos.items():
        print(f"\n entrenando {nombre}...")
        modelo.fit(X_train, y_train)

        #predicciones
        y_pred = modelo.predict(X_test)
        y_proba = modelo.predict_proba(X_test)[:, 1]

        #métricas
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc_roc = roc_auc_score(y_test, y_proba)

        resultados.append({
            'Modelo': nombre,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'AUC-ROC': auc_roc
        })

        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
        print(f"  AUC-ROC: {auc_roc:.4f}")

    return pd.DataFrame(resultados), modelos

df_resultados_clf, modelos_clf = entrenar_modelos_clasificacion(
    X_train_scaled, X_test_scaled, y_clf_train, y_clf_test
)

print("\n" + "="*60)
print("resultados de clasificación:")
print("="*60)
print(df_resultados_clf.to_string(index=False))

In [ ]:
# modelos de regresión
def entrenar_modelos_regresion(X_train, X_test, y_train, y_test):
    modelos = {
        'Regresión Lineal': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
    }

    resultados = []

    for nombre, modelo in modelos.items():
        print(f"\nEntrenando {nombre}...")
        modelo.fit(X_train, y_train)

        #predicciones
        y_pred = modelo.predict(X_test)

        #métricas
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        resultados.append({
            'Modelo': nombre,
            'MAE (días)': mae,
            'RMSE (días)': rmse,
            'R²': r2
        })

        print(f"  MAE: {mae:.2f} días")
        print(f"  RMSE: {rmse:.2f} días")
        print(f"  R²: {r2:.3f}")

    return pd.DataFrame(resultados), modelos

df_resultados_reg, modelos_reg = entrenar_modelos_regresion(
    X_train_scaled, X_test_scaled, y_reg_train, y_reg_test
)

print("\n" + "="*60)
print("resultados de regresión")
print("="*60)
print(df_resultados_reg.to_string(index=False))

In [ ]:
# analisis por categoria

def analisis_por_categoria(df_model, modelo_clf, scaler, features):
    #cargar categorías si están disponibles
    try:
        categorias = cargar_categorias('US_category_id.json')
    except:
        categorias = {i: f'Cat_{i}' for i in df_model['category_id'].unique()}

    resultados_cat = []

    for cat_id in df_model['category_id'].unique():
        df_cat = df_model[df_model['category_id'] == cat_id]

        if len(df_cat) < 100:  #minimo 100 muestras
            continue

        X_cat = df_cat[features]
        y_cat = df_cat['viral']

        X_cat_scaled = scaler.transform(X_cat)
        y_pred = modelo_clf.predict(X_cat_scaled)

        precision = precision_score(y_cat, y_pred, zero_division=0)
        recall = recall_score(y_cat, y_pred, zero_division=0)
        f1 = f1_score(y_cat, y_pred, zero_division=0)

        cat_nombre = categorias.get(cat_id, f'Categoría {cat_id}')

        resultados_cat.append({
            'Categoría': cat_nombre,
            'Precisión': precision,
            'Recall': recall,
            'F1-Score': f1,
            'Soporte': len(df_cat)
        })

    df_cat_resultados = pd.DataFrame(resultados_cat)
    df_cat_resultados = df_cat_resultados.sort_values('F1-Score', ascending=False)

    return df_cat_resultados

#features utilizados
features = [
    'category_id', 'publish_hour', 'publish_dayofweek', 'publish_month',
    'log_views', 'log_likes', 'log_dislikes', 'log_comment_count',
    'engagement_rate', 'like_ratio', 'comments_disabled', 'ratings_disabled'
]

#usamos el mejor modelo
mejor_modelo_clf = modelos_clf['Gradient Boosting']
df_cat_resultados = analisis_por_categoria(df_model, mejor_modelo_clf, scaler, features)

print("\n" + "="*60)
print("DESEMPEÑO POR CATEGORÍA")
print("="*60)
print(df_cat_resultados.to_string(index=False))

In [ ]:
#analisis por pais

def analisis_por_pais(df):
    dias_por_pais = df.groupby('country')['days_to_trending'].mean().sort_values()

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(dias_por_pais.index, dias_por_pais.values, color='teal', edgecolor='black')
    ax.set_xlabel('Días Promedio para Alcanzar Tendencias')
    ax.set_ylabel('País')
    ax.set_title('Días Promedio para Alcanzar Tendencias por País')

    #añadir valores
    for bar, val in zip(bars, dias_por_pais.values):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=10)

    plt.tight_layout()
    plt.savefig('dias_por_pais.png', dpi=150)
    plt.show()

    return dias_por_pais

if df['country'].nunique() > 1:
    dias_por_pais = analisis_por_pais(df)
else:
    print("\nSolo hay un país en el dataset actual.")

In [ ]:
# visualizaciones

def visualizar_resultados_finales(df_clf, df_reg):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    #grafica de clasificación
    ax1 = axes[0]
    x = np.arange(len(df_clf))
    width = 0.15

    ax1.bar(x - 2*width, df_clf['Accuracy'], width, label='Accuracy', color='steelblue')
    ax1.bar(x - width, df_clf['Precision'], width, label='Precision', color='green')
    ax1.bar(x, df_clf['Recall'], width, label='Recall', color='orange')
    ax1.bar(x + width, df_clf['F1-Score'], width, label='F1-Score', color='red')
    ax1.bar(x + 2*width, df_clf['AUC-ROC'], width, label='AUC-ROC', color='purple')

    ax1.set_ylabel('Puntuación')
    ax1.set_title('Comparación de Modelos de Clasificación')
    ax1.set_xticks(x)
    ax1.set_xticklabels(df_clf['Modelo'], rotation=15, ha='right')
    ax1.legend(loc='lower right')
    ax1.set_ylim(0, 1)

    #grafica de regresión
    ax2 = axes[1]
    x = np.arange(len(df_reg))

    ax2_twin = ax2.twinx()

    bars1 = ax2.bar(x - 0.2, df_reg['MAE (días)'], 0.4, label='MAE', color='coral')
    bars2 = ax2.bar(x + 0.2, df_reg['RMSE (días)'], 0.4, label='RMSE', color='skyblue')
    line = ax2_twin.plot(x, df_reg['R²'], 'go-', linewidth=2, markersize=8, label='R²')

    ax2.set_ylabel('Error (días)')
    ax2_twin.set_ylabel('R²')
    ax2.set_title('Comparación de Modelos de Regresión')
    ax2.set_xticks(x)
    ax2.set_xticklabels(df_reg['Modelo'], rotation=15, ha='right')

    #combinar leyendas
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.tight_layout()
    plt.savefig('resultados_finales.png', dpi=150)
    plt.show()

visualizar_resultados_finales(df_resultados_clf, df_resultados_reg)

In [ ]:
# resumen final

print("\n" + "="*70)
print("RESUMEN DEL ANÁLISIS")
print("="*70)
print(f"""
Dataset: Trending YouTube Video Statistics (US)
Total de registros analizados: {len(df_model)}

Mejores Resultados:

Clasificación (Predecir si video supera 1M de vistas):
   Mejor modelo: Gradient Boosting
   - Accuracy: {df_resultados_clf.loc[df_resultados_clf['Modelo']=='Gradient Boosting', 'Accuracy'].values[0]:.4f}
   - AUC-ROC: {df_resultados_clf.loc[df_resultados_clf['Modelo']=='Gradient Boosting', 'AUC-ROC'].values[0]:.4f}

Regresión (Predecir días hasta tendencias):
   Mejor modelo: Gradient Boosting
   - MAE: {df_resultados_reg.loc[df_resultados_reg['Modelo']=='Gradient Boosting', 'MAE (días)'].values[0]:.2f} días
   - R²: {df_resultados_reg.loc[df_resultados_reg['Modelo']=='Gradient Boosting', 'R²'].values[0]:.3f}

Patron estacional:
   - Mayor actividad: Miércoles (índice: {indice_estacional[2]:.2f})
   - Menor actividad: Fines de semana

Imagenes generadas:
   - distribucion_popularidad.png
   - patrones_estacionales.png
   - resultados_finales.png
""")